In [38]:
# import libraries
import pandas as pd
import sqlite3

In [2]:
# load in flat file as df
df = pd.read_csv("../Data/bookstore_flat.csv")

In [3]:
# inspect the df
df.head()

,transaction_id,transaction_date,member_id,member_first_name,member_last_name,member_email,member_tier,member_join_date,member_city,member_state,...,author_first,author_last,publisher_name,publisher_city,publisher_country,publication_year,list_price,quantity,discount_percent,sale_total
0,1,2019-10-05,260,Casey,Jackson,casey.jackson260@email.com,Platinum,2018-12-09,New York,WA,...,Celeste,Ng,Macmillan Publishers,New York,USA,2006,19.23,3,20,46.14
1,2,2019-12-22,292,Marcus,Walker,marcus.walker292@email.com,Platinum,2020-06-11,Portland,CO,...,Erik,Larson,Hachette Book Group,New York,USA,2008,28.46,1,0,28.46
2,3,2019-03-09,30,Bailey,Anderson,bailey.anderson30@email.com,Bronze,2018-05-26,Austin,TX,...,Isabel,Allende,Scholastic,New York,USA,2020,34.94,1,5,33.19
3,4,2020-02-12,185,Ethan,King,ethan.king185@email.com,Bronze,2018-01-05,Portland,CA,...,Jhumpa,Lahiri,Penguin Random House,New York,USA,2012,31.17,2,0,62.34
4,5,2021-03-23,238,Kenji,Scott,kenji.scott238@email.com,Gold,2022-10-28,Coral Gables,OR,...,N.K.,Jemisin,Hachette Book Group,New York,USA,2007,19.10,1,15,16.23


In [4]:
19.23*3*.8

46.152

In [5]:
df.columns

Index(['transaction_id', 'transaction_date', 'member_id', 'member_first_name',
       'member_last_name', 'member_email', 'member_tier', 'member_join_date',
       'member_city', 'member_state', 'store_name', 'store_city',
       'store_state', 'store_region', 'book_id', 'isbn', 'title', 'genre',
       'format', 'author_first', 'author_last', 'publisher_name',
       'publisher_city', 'publisher_country', 'publication_year', 'list_price',
       'quantity', 'discount_percent', 'sale_total'],
      dtype='str')

Use this markdown cell to write notes needed to design the new relational db schema

- transactions = ['transaction_id', 'transaction_date', 'member_id', 'book_id', 'store_id', 'quantity', 'discount_percent'] ****store_id



- books = ['book_id', 'isbn', 'title', 'genre_id','format', 'author_id', 'publisher_id', 'list_price', 'publication_year'] ***genre_id, author_id, publisher_id


- members = ['member_id', 'member_first_name', 'member_last_name', 'member_email', 'member_tier', 'member_join_date', 'member_city', 'member_state']
- stores = ['store_name', 'store_city', 'store_state', 'store_region'] *** store_id?
- publishers = ['publisher_name','publisher_city', 'publisher_country'] ***publisher_id
- genres = ?? we have to look up unique -> ['genre_id', 'name']
- authors = ['author_id', 'author_first', 'author_last'] **author_id




In [6]:
# filter the df into tables matching the schema in the markdown cell above

genres_df = df[['genre']].drop_duplicates().reset_index(drop=True)

In [7]:
genres_df

,genre
0,Biography
1,History
2,Science Fiction
3,Fiction
4,Mystery
5,Fantasy
6,Self-Help


In [8]:
genres_df.insert(0, 'genre_id', range(1, len(genres_df) + 1))

In [9]:
genres_df = genres_df.rename(columns = {'genre':'name'})
genres_df

,genre_id,name
0,1,Biography
1,2,History
2,3,Science Fiction
3,4,Fiction
4,5,Mystery
5,6,Fantasy
6,7,Self-Help


In [10]:
authors_df = df[['author_first', 'author_last']].drop_duplicates().reset_index(drop=True)
authors_df.head(20)

,author_first,author_last
0,Celeste,Ng
1,Erik,Larson
2,Isabel,Allende
3,Jhumpa,Lahiri
4,N.K.,Jemisin
5,Patrick,Rothfuss
6,Ocean,Vuong
7,Colson,Whitehead
8,Zadie,Smith
9,Roxane,Gay


In [11]:
authors_df.insert(0, 'authors_id', range(1, len(authors_df) + 1))
authors_df.head()

,authors_id,author_first,author_last
0,1,Celeste,Ng
1,2,Erik,Larson
2,3,Isabel,Allende
3,4,Jhumpa,Lahiri
4,5,N.K.,Jemisin


In [21]:
authors_df = authors_df.rename(columns={'authors_id':'author_id', 'author_first':'first_name', 'author_last':'last_name'})
authors_df.head()

,author_id,first_name,last_name
0,1,Celeste,Ng
1,2,Erik,Larson
2,3,Isabel,Allende
3,4,Jhumpa,Lahiri
4,5,N.K.,Jemisin


In [22]:
publisher_df = df[['publisher_name','publisher_city', 'publisher_country']].drop_duplicates().reset_index(drop=True)
publisher_df.insert(0, 'publisher_id', range(1, len(publisher_df) + 1))
publisher_df.head()

,publisher_id,publisher_name,publisher_city,publisher_country
0,1,Macmillan Publishers,New York,USA
1,2,Hachette Book Group,New York,USA
2,3,Scholastic,New York,USA
3,4,Penguin Random House,New York,USA
4,5,Bloomsbury,London,UK


In [23]:
stores_df = df[['store_name', 'store_city', 'store_state', 'store_region']].drop_duplicates().reset_index(drop=True)
stores_df.insert(0, 'store_id', range(1, len(stores_df) + 1))
stores_df.head()

,store_id,store_name,store_city,store_state,store_region
0,1,Prairie Lights,Iowa City,IA,Midwest
1,2,Powell's Books,Portland,OR,Pacific
2,3,BookPeople,Austin,TX,South
3,4,Left Bank Books,St. Louis,MO,Midwest
4,5,Malaprop's Bookstore,Asheville,NC,South


In [24]:
members_df = df[['member_id', 'member_first_name', 'member_last_name', 'member_email', 'member_tier', 'member_join_date', 'member_city', 'member_state']].drop_duplicates()
members_df.head(50)


,member_id,member_first_name,member_last_name,member_email,member_tier,member_join_date,member_city,member_state
0,260,Casey,Jackson,casey.jackson260@email.com,Platinum,2018-12-09,New York,WA
1,292,Marcus,Walker,marcus.walker292@email.com,Platinum,2020-06-11,Portland,CO
2,30,Bailey,Anderson,bailey.anderson30@email.com,Bronze,2018-05-26,Austin,TX
3,185,Ethan,King,ethan.king185@email.com,Bronze,2018-01-05,Portland,CA
4,238,Kenji,Scott,kenji.scott238@email.com,Gold,2022-10-28,Coral Gables,OR
5,16,Hector,Green,hector.green16@email.com,Gold,2023-03-16,Coral Gables,FL
6,91,Ethan,Walker,ethan.walker91@email.com,Silver,2018-05-25,Cambridge,CA
7,84,Kenji,White,kenji.white84@email.com,Silver,2022-04-20,Cambridge,CA
8,146,Alex,Davis,alex.davis146@email.com,Gold,2020-07-19,Menlo Park,MA
9,293,Iris,Jackson,iris.jackson293@email.com,Platinum,2016-08-12,Portland,IA


In [25]:
members_df['member_id'].value_counts()

member_id
260    1
292    1
30     1
185    1
238    1
      ..
164    1
143    1
39     1
35     1
101    1
Name: count, Length: 300, dtype: int64

In [26]:
books_df = df[['book_id', 'isbn', 'title', 'genre','format', 'author_first', 'author_last', 'publisher_name', 'list_price', 'publication_year']].drop_duplicates()
books_df.head()

,book_id,isbn,title,genre,format,author_first,author_last,publisher_name,list_price,publication_year
0,182,978-529-93748-89-7,The Long Road Vol. 6,Biography,Paperback,Celeste,Ng,Macmillan Publishers,19.23,2006
1,226,978-119-67807-50-6,The Lost Century Vol. 2,History,Hardcover,Erik,Larson,Hachette Book Group,28.46,2008
2,215,978-578-75373-49-7,Empire Rising Vol. 7,History,Paperback,Isabel,Allende,Scholastic,34.94,2020
3,236,978-856-14888-32-6,Turning Points Vol. 4,History,Hardcover,Jhumpa,Lahiri,Penguin Random House,31.17,2012
4,211,978-161-30719-66-6,Empire Rising Vol. 3,History,Audiobook,N.K.,Jemisin,Hachette Book Group,19.10,2007


In [27]:
books_df =  books_df.merge(genres_df[['genre_id', 'name']], left_on='genre', right_on= 'name', how='left')
books_df.head()

,book_id,isbn,title,genre,format,author_first,author_last,publisher_name,list_price,publication_year,genre_id,name
0,182,978-529-93748-89-7,The Long Road Vol. 6,Biography,Paperback,Celeste,Ng,Macmillan Publishers,19.23,2006,1,Biography
1,226,978-119-67807-50-6,The Lost Century Vol. 2,History,Hardcover,Erik,Larson,Hachette Book Group,28.46,2008,2,History
2,215,978-578-75373-49-7,Empire Rising Vol. 7,History,Paperback,Isabel,Allende,Scholastic,34.94,2020,2,History
3,236,978-856-14888-32-6,Turning Points Vol. 4,History,Hardcover,Jhumpa,Lahiri,Penguin Random House,31.17,2012,2,History
4,211,978-161-30719-66-6,Empire Rising Vol. 3,History,Audiobook,N.K.,Jemisin,Hachette Book Group,19.10,2007,2,History


In [28]:
books_df = books_df.drop(columns=['genre', 'name'])
books_df.head()

,book_id,isbn,title,format,author_first,author_last,publisher_name,list_price,publication_year,genre_id
0,182,978-529-93748-89-7,The Long Road Vol. 6,Paperback,Celeste,Ng,Macmillan Publishers,19.23,2006,1
1,226,978-119-67807-50-6,The Lost Century Vol. 2,Hardcover,Erik,Larson,Hachette Book Group,28.46,2008,2
2,215,978-578-75373-49-7,Empire Rising Vol. 7,Paperback,Isabel,Allende,Scholastic,34.94,2020,2
3,236,978-856-14888-32-6,Turning Points Vol. 4,Hardcover,Jhumpa,Lahiri,Penguin Random House,31.17,2012,2
4,211,978-161-30719-66-6,Empire Rising Vol. 3,Audiobook,N.K.,Jemisin,Hachette Book Group,19.10,2007,2


In [29]:
books_df = books_df.merge(authors_df[['author_id', 'first_name', 'last_name']], left_on=['author_first', 'author_last'], right_on=['first_name', 'last_name'] ,how='left')
books_df.head()

,book_id,isbn,title,format,author_first,author_last,publisher_name,list_price,publication_year,genre_id,author_id,first_name,last_name
0,182,978-529-93748-89-7,The Long Road Vol. 6,Paperback,Celeste,Ng,Macmillan Publishers,19.23,2006,1,1,Celeste,Ng
1,226,978-119-67807-50-6,The Lost Century Vol. 2,Hardcover,Erik,Larson,Hachette Book Group,28.46,2008,2,2,Erik,Larson
2,215,978-578-75373-49-7,Empire Rising Vol. 7,Paperback,Isabel,Allende,Scholastic,34.94,2020,2,3,Isabel,Allende
3,236,978-856-14888-32-6,Turning Points Vol. 4,Hardcover,Jhumpa,Lahiri,Penguin Random House,31.17,2012,2,4,Jhumpa,Lahiri
4,211,978-161-30719-66-6,Empire Rising Vol. 3,Audiobook,N.K.,Jemisin,Hachette Book Group,19.10,2007,2,5,N.K.,Jemisin


In [30]:
books_df = books_df.drop(columns=['first_name', 'last_name'])
books_df.head()

,book_id,isbn,title,format,author_first,author_last,publisher_name,list_price,publication_year,genre_id,author_id
0,182,978-529-93748-89-7,The Long Road Vol. 6,Paperback,Celeste,Ng,Macmillan Publishers,19.23,2006,1,1
1,226,978-119-67807-50-6,The Lost Century Vol. 2,Hardcover,Erik,Larson,Hachette Book Group,28.46,2008,2,2
2,215,978-578-75373-49-7,Empire Rising Vol. 7,Paperback,Isabel,Allende,Scholastic,34.94,2020,2,3
3,236,978-856-14888-32-6,Turning Points Vol. 4,Hardcover,Jhumpa,Lahiri,Penguin Random House,31.17,2012,2,4
4,211,978-161-30719-66-6,Empire Rising Vol. 3,Audiobook,N.K.,Jemisin,Hachette Book Group,19.10,2007,2,5


In [31]:
trans_df = df[['transaction_id', 'transaction_date', 'member_id', 'book_id', 'store_name', 'quantity', 'discount_percent']].drop_duplicates()
trans_df.head()

,transaction_id,transaction_date,member_id,book_id,store_name,quantity,discount_percent
0,1,2019-10-05,260,182,Prairie Lights,3,20
1,2,2019-12-22,292,226,Powell's Books,1,0
2,3,2019-03-09,30,215,BookPeople,1,5
3,4,2020-02-12,185,236,Left Bank Books,2,0
4,5,2021-03-23,238,211,Malaprop's Bookstore,1,15


In [32]:
trans_df = trans_df.merge(stores_df[['store_id', 'store_name']], on='store_name', how='left')
trans_df.head()

,transaction_id,transaction_date,member_id,book_id,store_name,quantity,discount_percent,store_id
0,1,2019-10-05,260,182,Prairie Lights,3,20,1
1,2,2019-12-22,292,226,Powell's Books,1,0,2
2,3,2019-03-09,30,215,BookPeople,1,5,3
3,4,2020-02-12,185,236,Left Bank Books,2,0,4
4,5,2021-03-23,238,211,Malaprop's Bookstore,1,15,5


In [33]:
trans_df = trans_df.drop(columns=['store_name'])
trans_df.head()

,transaction_id,transaction_date,member_id,book_id,quantity,discount_percent,store_id
0,1,2019-10-05,260,182,3,20,1
1,2,2019-12-22,292,226,1,0,2
2,3,2019-03-09,30,215,1,5,3
3,4,2020-02-12,185,236,2,0,4
4,5,2021-03-23,238,211,1,15,5


In [37]:
# save each of the new dfs as their own csv files
genres_df.to_csv('../Data/genres.csv')
books_df.to_csv('../Data/books.csv')
authors_df.to_csv('../Data/authors.csv')
publisher_df.to_csv('../Data/publisher.csv')
stores_df.to_csv('../Data/stores.csv')
members_df.to_csv('../Data/members.csv')
trans_df.to_csv('../Data/transactions.csv')

In [40]:
# create the database and load all new dfs as tables. make sure to save the db in the data folder. 
conn = sqlite3.connect('../Data/bookstore.db')
genres_df.to_sql('genres', conn, if_exists = 'replace', index=False)
authors_df.to_sql('authors', conn, if_exists = 'replace', index=False)
books_df.to_sql('books', conn, if_exists = 'replace', index=False)
members_df.to_sql('members', conn, if_exists = 'replace', index=False)
publisher_df.to_sql('publishers', conn, if_exists = 'replace', index=False)
stores_df.to_sql('stores', conn, if_exists = 'replace', index=False)
trans_df.to_sql('transactions', conn, if_exists = 'replace', index=False)


5200

In [41]:
conn.close()